In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from copy import copy
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill


input_file = DATA_ROOT / "interdependence_paper_supplementary_data_new_with_dependence_scores.xlsx"
output_file = DATA_ROOT / "interdependence_paper_supplementary_data_final.xlsx"

assert input_file.exists(), input_file
assert input_file.resolve() != output_file.resolve()


In [ ]:
def clean_text(value):
    if value is None:
        return ""
    text = str(value).replace("\xa0", " ").strip()
    if text.casefold() in {"nan", "<na>", "none"}:
        return ""
    return " ".join(text.split())


def is_used_in_analysis(value):
    if value is True or value == 1:
        return True
    return clean_text(value).casefold() in {"true", "yes", "y", "1"}


HEADER_ACRONYMS = {
    "asjc": "ASJC",
    "doi": "DOI",
    "eu": "EU",
    "iso2": "ISO2",
    "ngo": "NGO",
    "us": "US",
    "url": "URL",
}


def capitalize_header(value):
    words = clean_text(value).replace("_", " ").split()
    return " ".join(
        HEADER_ACRONYMS.get(word.casefold(), word.capitalize())
        for word in words
    )


def header_key(value):
    return clean_text(value).replace(" ", "_").casefold()


def split_semicolon_values(value):
    values = []
    seen = set()
    for item in clean_text(value).split(";"):
        item = clean_text(item)
        key = item.casefold()
        if item and key not in seen:
            seen.add(key)
            values.append(item)
    return values


def build_unique_lookup(records, key_column, value_column, label, fallback_to_key=False):
    lookup = {}
    for record in records:
        key = clean_text(record.get(key_column))
        if not key:
            continue
        value = clean_text(record.get(value_column))
        if not value and fallback_to_key:
            value = key
        if key in lookup and lookup[key] != value:
            raise ValueError(
                f"Conflicting {label} values for {key!r}: "
                f"{lookup[key]!r} and {value!r}"
            )
        lookup[key] = value
    return lookup


def map_visualization_names(value, lookup, label):
    text = clean_text(value)
    if not text:
        return ""
    if text in lookup:
        return lookup[text]
    mapped = []
    missing = []
    for item in split_semicolon_values(text):
        if item not in lookup:
            missing.append(item)
        else:
            mapped.extend(split_semicolon_values(lookup[item]))
    if missing:
        raise KeyError(f"Missing {label} for: {missing}")
    return "; ".join(dict.fromkeys(mapped))


def map_final_holder_type(holder_names, holder_type_lookup):
    holder_types = []
    missing = []
    for holder_name in split_semicolon_values(holder_names):
        holder_type = clean_text(holder_type_lookup.get(holder_name))
        if not holder_type:
            missing.append(holder_name)
        else:
            holder_types.extend(split_semicolon_values(holder_type))
    if missing:
        raise KeyError(f"Missing holder_type_new classification for: {missing}")
    unique_types = list(dict.fromkeys(holder_types))
    if len(unique_types) > 1:
        return "hybrid"
    return unique_types[0] if unique_types else ""


In [ ]:
source_book = pd.ExcelFile(input_file)
required_sheets = {
    "paper_source_data",
    "standardized_sources",
    "visualization_names",
    "unique_institutions",
    "Sheet1",
}
missing_sheets = required_sheets - set(source_book.sheet_names)
if missing_sheets:
    raise KeyError(f"Missing required sheets: {sorted(missing_sheets)}")

paper_data = pd.read_excel(
    source_book, sheet_name="paper_source_data", keep_default_na=False
)
standardized_sources = pd.read_excel(
    source_book, sheet_name="standardized_sources", keep_default_na=False
)
visualization_names = pd.read_excel(
    source_book, sheet_name="visualization_names", keep_default_na=False
)
unique_institutions = pd.read_excel(
    source_book, sheet_name="unique_institutions", keep_default_na=False
)

# Keep only rows that were used in the analysis, then remove the raw names.
if "used_in_analysis" not in paper_data.columns:
    raise KeyError("used_in_analysis not found in paper_source_data")
original_paper_rows = len(paper_data)
paper_data = paper_data.loc[
    paper_data["used_in_analysis"].map(is_used_in_analysis)
].copy()
paper_data.drop(columns=["raw_extracted_source_names"], inplace=True)
retained_paper_rows = len(paper_data)
removed_paper_rows = original_paper_rows - retained_paper_rows

# Build the reviewed holder-type lookup.
holder_type_lookup = build_unique_lookup(
    unique_institutions.to_dict("records"),
    key_column="data_holder",
    value_column="holder_type_new",
    label="holder-type classification",
)

# Apply the holder-type changes used in the final analysis. EU institutions
# classified as international public are treated as national public.
eu_mask = (
    unique_institutions["EU"].map(is_used_in_analysis)
    & unique_institutions["holder_type_new"]
        .map(clean_text)
        .str.casefold()
        .eq("international public")
)
eu_public_holders = {
    clean_text(value)
    for value in unique_institutions.loc[eu_mask, "data_holder"]
    if clean_text(value)
}
for holder_name in eu_public_holders:
    holder_type_lookup[holder_name] = "national public"

# Apply the reviewed Missouri Botanical Garden override used downstream.
if "Missouri Botanical Garden" not in holder_type_lookup:
    raise KeyError("Missouri Botanical Garden is missing from the holder lookup")
holder_type_lookup["Missouri Botanical Garden"] = "hybrid"

resource_visualization_lookup = build_unique_lookup(
    visualization_names.to_dict("records"),
    key_column="standardized_data_resource_name",
    value_column="data_resource_for_visualization",
    label="resource visualization name",
    fallback_to_key=True,
)
holder_visualization_lookup = build_unique_lookup(
    visualization_names.to_dict("records"),
    key_column="standardized_data_holder_name",
    value_column="data_holder_for_visualization",
    label="holder visualization name",
    fallback_to_key=True,
)

required_standardized_columns = {
    "raw_name",
    "standardized_data_resource_name",
    "standardized_data_holder_name",
    "standardized_data_holder_type",
}
missing_columns = required_standardized_columns - set(standardized_sources.columns)
if missing_columns:
    raise KeyError(
        f"standardized_sources is missing columns: {sorted(missing_columns)}"
    )

standardized_sources["data_resource_for_visualization"] = standardized_sources[
    "standardized_data_resource_name"
].map(
    lambda value: map_visualization_names(
        value, resource_visualization_lookup, "resource visualization name"
    )
)
standardized_sources["data_holder_for_visualization"] = standardized_sources[
    "standardized_data_holder_name"
].map(
    lambda value: map_visualization_names(
        value, holder_visualization_lookup, "holder visualization name"
    )
)
standardized_sources["standardized_data_holder_type"] = standardized_sources[
    "standardized_data_holder_name"
].map(lambda value: map_final_holder_type(value, holder_type_lookup))

# Remove the raw-name fields, including the count that only described them.
columns_to_remove = [
    column
    for column in standardized_sources.columns
    if header_key(column) in {"raw_name", "raw_name_document_count"}
]
removed_column_keys = {header_key(column) for column in columns_to_remove}
if "raw_name" not in removed_column_keys:
    raise KeyError("raw_name was not found in standardized_sources")
if "raw_name_document_count" not in removed_column_keys:
    raise KeyError(
        "Raw Name Document Count was not found in standardized_sources"
    )
standardized_sources.drop(columns=columns_to_remove, inplace=True)

# Remove rows that became duplicates after the raw-name fields were deleted.
standardized_rows_before_deduplication = len(standardized_sources)
standardized_sources = standardized_sources.drop_duplicates().reset_index(
    drop=True
)
removed_standardized_duplicates = (
    standardized_rows_before_deduplication - len(standardized_sources)
)

removed_sheets = {"Sheet1", "unique_institutions", "visualization_names"}
final_sheets = {}
for sheet_name in source_book.sheet_names:
    if sheet_name in removed_sheets:
        continue
    if sheet_name == "paper_source_data":
        final_sheets[sheet_name] = paper_data
    elif sheet_name == "standardized_sources":
        final_sheets[sheet_name] = standardized_sources
    else:
        final_sheets[sheet_name] = pd.read_excel(
            source_book, sheet_name=sheet_name, keep_default_na=False
        )
source_book.close()

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, frame in final_sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name, index=False)

# Capitalize and bold every data-sheet header row and enable filtering.
workbook = load_workbook(output_file)
for worksheet in workbook.worksheets:
    for cell in worksheet[1]:
        cell.value = capitalize_header(cell.value)
        bold_font = copy(cell.font)
        bold_font.bold = True
        cell.font = bold_font
    worksheet.auto_filter.ref = worksheet.dimensions
    worksheet.freeze_panes = "A2"

# Add a publication-facing guide as the first worksheet.
readme = workbook.create_sheet("ReadMe", 0)
readme.sheet_view.showGridLines = False
readme.column_dimensions["A"].width = 31
readme.column_dimensions["B"].width = 105

readme["A1"] = "Supplementary Data ReadMe"
readme.merge_cells("A1:B1")
readme["A1"].font = Font(size=16, bold=True, color="1F4E78")
readme["A1"].alignment = Alignment(vertical="center")
readme.row_dimensions[1].height = 28

readme["A3"] = "Purpose"
readme["A4"] = (
    "This workbook contains the supplementary data used in the analysis. "
    "It provides the paper-level analytical dataset, standardized source "
    "metadata, data-holder dependence scores, and country-level source-use "
    "summaries. Detailed definitions, data-processing procedures, and "
    "analytical methods are provided in the Methods Appendix."
)
readme.merge_cells("A4:B4")
readme["A4"].alignment = Alignment(wrap_text=True, vertical="top")
readme.row_dimensions[4].height = 58

readme["A6"] = "Worksheet"
readme["B6"] = "Contents"
worksheet_descriptions = [
    (
        "Paper Source Data",
        "Paper-level records included in the analysis, with bibliographic "
        "information, author-affiliation countries, subject classifications, "
        "and standardized data-resource and data-holder names.",
    ),
    (
        "Standardized Sources",
        "Standardized data-resource and data-holder records. The sheet includes "
        "names used in figures, data-holder classifications, holder-country "
        "codes, document counts, and URLs where available.",
    ),
    (
        "Average Dependence Scores",
        "Summary of average dependence on each data holder across countries, "
        "together with the number of unique papers and countries associated "
        "with each holder.",
    ),
    (
        "Country Year Source Shares",
        "Annual source-use shares by user country and publication year, "
        "including domestic, external, international-public, private, and "
        "NGO-held data sources.",
    ),
    (
        "Country Average Source Shares",
        "Country-level source-use shares averaged across the study period, "
        "with the number of included papers and years with available data.",
    ),
]
for row_number, (sheet_name, description) in enumerate(
    worksheet_descriptions, start=7
):
    readme.cell(row=row_number, column=1, value=sheet_name)
    readme.cell(row=row_number, column=2, value=description)
    readme.cell(row=row_number, column=1).font = Font(bold=True)
    readme.cell(row=row_number, column=1).alignment = Alignment(vertical="top")
    readme.cell(row=row_number, column=2).alignment = Alignment(
        wrap_text=True, vertical="top"
    )
    readme.row_dimensions[row_number].height = 42

readme["A13"] = "General Notes"
readme["A14"] = "Filtering"
readme["B14"] = "Each data worksheet can be filtered using its header row."
readme["A15"] = "Multiple values"
readme["B15"] = "Multiple values within a cell are separated by semicolons."
readme["A16"] = "Missing values"
readme["B16"] = (
    "Blank cells indicate that a value was unavailable or not applicable."
)
readme["A17"] = "Country codes"
readme["B17"] = "Country fields use ISO 3166-1 alpha-2 codes unless noted otherwise."
readme["A18"] = "Percentages"
readme["B18"] = "Percentage columns are expressed on a 0–100 scale."
readme["A19"] = "Methods"
readme["B19"] = (
    "Consult the Methods Appendix for variable definitions, inclusion criteria, "
    "standardization procedures, and analytical details."
)
for row_number in range(14, 20):
    readme.cell(row=row_number, column=1).font = Font(bold=True)
    readme.cell(row=row_number, column=1).alignment = Alignment(vertical="top")
    readme.cell(row=row_number, column=2).alignment = Alignment(
        wrap_text=True, vertical="top"
    )
    readme.row_dimensions[row_number].height = 30

section_fill = PatternFill("solid", fgColor="1F4E78")
for section_row in [3, 6, 13]:
    for cell in readme[section_row]:
        cell.fill = section_fill
        cell.font = Font(bold=True, color="FFFFFF")
        cell.alignment = Alignment(vertical="center")
    readme.row_dimensions[section_row].height = 22

workbook.save(output_file)
workbook.close()

print(f"Retained paper_source_data rows: {retained_paper_rows:,}")
print(f"Removed unused paper_source_data rows: {removed_paper_rows:,}")
print(
    "EU holders changed from international public to national public: "
    f"{len(eu_public_holders):,}"
)
print(
    "Duplicate standardized-source rows removed: "
    f"{removed_standardized_duplicates:,}"
)
print(f"Saved final supplementary workbook: {output_file}")


In [ ]:
check_workbook = load_workbook(output_file, read_only=True, data_only=True)

for removed_sheet in ["Sheet1", "unique_institutions", "visualization_names"]:
    assert removed_sheet not in check_workbook.sheetnames

paper_headers = [header_key(cell.value) for cell in check_workbook["paper_source_data"][1]]
standardized_headers = [
    header_key(cell.value) for cell in check_workbook["standardized_sources"][1]
]
assert "raw_extracted_source_names" not in paper_headers
used_column = paper_headers.index("used_in_analysis") + 1
assert all(
    is_used_in_analysis(row[0])
    for row in check_workbook["paper_source_data"].iter_rows(
        min_row=2,
        min_col=used_column,
        max_col=used_column,
        values_only=True,
    )
)
assert "raw_name" not in standardized_headers
assert "raw_name_document_count" not in standardized_headers
assert "standardized_data_holder_type" in standardized_headers
assert "data_resource_for_visualization" in standardized_headers
assert "data_holder_for_visualization" in standardized_headers
standardized_rows = list(
    check_workbook["standardized_sources"].iter_rows(
        min_row=2, values_only=True
    )
)
assert len(standardized_rows) == len(set(standardized_rows))
assert check_workbook.sheetnames[0] == "ReadMe"
for worksheet in check_workbook.worksheets:
    assert all(cell.font.bold for cell in worksheet[1] if cell.value is not None)

check_workbook.close()
print("Validation passed.")
